# ML-04 — Search Intelligence Data Contract

**Lane Selected:** Refresh / Content Opportunity Scoring  
**Domain:** Applied Search Intelligence & Content Lifecycle Management  
**Dataset:** Starter Dataset (`data/raw/content_refresh_anonymized.csv`)  

This notebook defines the data contract, verifies key grain & availability facts with queries, constructs a 5-feature decision frame, and demonstrates a deliberate feature leakage experiment.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The 5 Plain-Words Data Contract Answers:

1. **Unit of Analysis (Grain):** **One row = One pseudonymized content item (`content_id`)**.
2. **Table(s) Used:** `data/raw/content_refresh_anonymized.csv` (a 30,000-row anonymized starter slice of FlyRank's central search warehouse).
3. **Time Window:** **Trailing 90 days** relative to snapshot collection (`impressions_90d`, `sessions_90d`, etc.).
4. **Target / Proxy Label:** `is_declining_label`, defined as `(trend_direction == 'down')`.
5. **Deliberately Excluded:** `trend_direction` and `trend_pct` from feature set (direct target leakage), plus pseudonym IDs (`content_id`, `client_id`, `url_hash_id`, `keyword_hash_id`) as non-learnable metadata.

In [2]:
# Display Data Contract Summary
import pandas as pd

contract_summary = {
    "1. Unit of Analysis": "1 row = 1 pseudonymized content item (content_id)",
    "2. Table Used": "data/raw/content_refresh_anonymized.csv (30,000 rows x 44 cols)",
    "3. Time Window": "Trailing 90-day observation window",
    "4. Target / Proxy": "is_declining_label = (trend_direction == 'down')",
    "5. Deliberately Excluded": "trend_direction, trend_pct (target leakage risks); pseudonym IDs"
}

pd.DataFrame([contract_summary]).T.rename(columns={0: "Data Contract Specification"})

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Bucket Classification:

1. **Features (Safe & Knowable at Decision Moment):**
   * `impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d` (Search & traffic volume)
   * `avg_position`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` (Performance & engagement rates)
   * `content_age_days`, `days_since_last_update`, `word_count`, `char_count` (Content metadata & freshness)
   * `competition_level`, `content_type`, `main_intent` (Categorical search context)

2. **Label / Proxy:**
   * `is_declining_label` (`trend_direction == 'down'`): The binary outcome being predicted.

3. **Context (Joining, Grouping, & Validation Splits):**
   * `content_id`: Unique row identifier (grain).
   * `client_id`: Client identifier (used for GroupKFold client-holdout splits).
   * `url_hash_id`, `keyword_hash_id`: Pseudonymized context for grouping and case studies.

4. **Excluded (Leakage & Privacy Controls):**
   * `trend_direction`, `trend_pct`: **EXCLUDED**. `trend_direction` is derived directly from `trend_pct`. Using either as a feature leaks the target answer directly into the input!
   * Raw URLs, Raw Query Text, Client Names: **EXCLUDED**. Scrambled prior to release for data safety.

In [4]:
# Field Bucket Mapping
field_buckets = [
    {"Field": "impressions_90d", "Bucket": "Feature", "Rationale": "Knowable prior to decision point"},
    {"Field": "sessions_90d", "Bucket": "Feature", "Rationale": "Knowable prior to decision point"},
    {"Field": "avg_position", "Bucket": "Feature", "Rationale": "Knowable prior to decision point"},
    {"Field": "ctr", "Bucket": "Feature", "Rationale": "Knowable prior to decision point"},
    {"Field": "content_age_days", "Bucket": "Feature", "Rationale": "Knowable prior to decision point"},
    {"Field": "is_declining_label", "Bucket": "Label / Proxy", "Rationale": "Target outcome to predict"},
    {"Field": "content_id", "Bucket": "Context", "Rationale": "Unique row identifier"},
    {"Field": "client_id", "Bucket": "Context", "Rationale": "Group identifier for client holdout split"},
    {"Field": "trend_pct", "Bucket": "Excluded", "Rationale": "Target leakage (directly derives is_declining_label)"},
    {"Field": "trend_direction", "Bucket": "Excluded", "Rationale": "Target leakage (directly defines is_declining_label)"}
]

pd.DataFrame(field_buckets)

,Field,Bucket,Rationale
0,impressions_90d,Feature,Knowable prior to decision point
1,sessions_90d,Feature,Knowable prior to decision point
2,avg_position,Feature,Knowable prior to decision point
3,ctr,Feature,Knowable prior to decision point
4,content_age_days,Feature,Knowable prior to decision point
5,is_declining_label,Label / Proxy,Target outcome to predict
6,content_id,Context,Unique row identifier
7,client_id,Context,Group identifier for client holdout split
8,trend_pct,Excluded,Target leakage (directly derives is_declining_...
9,trend_direction,Excluded,Target leakage (directly defines is_declining_...


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: Grain Check (Is `content_id` unique per row?)

In [6]:
import os
import pandas as pd
import numpy as np

# Load dataset
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

# Query 1: Verify Grain (Duplicates per content_id)
duplicate_check = df.groupby('content_id').size().reset_index(name='count').query('count > 1')

print("--- QUERY 1: GRAIN VERIFICATION ---")
print(f"Total Rows in Dataset: {len(df):,}")
print(f"Unique content_id Count: {df['content_id'].nunique():,}")
print(f"Duplicate content_id Rows Found: {len(duplicate_check)}")
if len(duplicate_check) == 0:
    print("VERDICT: Grain holds strictly! 1 Row = 1 Pseudonymized Content Item.")

--- QUERY 1: GRAIN VERIFICATION ---
Total Rows in Dataset: 30,000
Unique content_id Count: 30,000
Duplicate content_id Rows Found: 0
VERDICT: Grain holds strictly! 1 Row = 1 Pseudonymized Content Item.


### Query 2: Row Counts, Client Distribution, & Age Windows

In [8]:
# Query 2: Counts, Age Span, and Client Counts
print("--- QUERY 2: COUNTS AND WINDOW SPAN ---")
print(f"Total Clients: {df['client_id'].nunique()}")
print(f"Content Age Min (Days): {df['content_age_days'].min()}")
print(f"Content Age Max (Days): {df['content_age_days'].max()}")
print(f"Content Age Median (Days): {df['content_age_days'].median():.1f}")
print(f"Days Since Last Update Range: {df['days_since_last_update'].min()} to {df['days_since_last_update'].max()} days")

--- QUERY 2: COUNTS AND WINDOW SPAN ---
Total Clients: 32
Content Age Min (Days): 90
Content Age Max (Days): 564
Content Age Median (Days): 236.0
Days Since Last Update Range: 1 to 373 days


### Query 3: Availability & Active Page Filters

In [10]:
# Query 3: Availability Checks
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Filter active pages with minimum volume & age
active_mask = (df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)
active_df = df[active_mask].copy()

print("--- QUERY 3: AVAILABILITY & SURVIVING ROWS ---")
print(f"Raw Dataset Rows: {len(df):,}")
print(f"Active Slice (impressions_90d > 0 AND content_age_days >= 90): {len(active_df):,}")
print(f"Retention Rate: {len(active_df) / len(df) * 100:.2f}%")
print(f"Active Slice Declining Label Ratio: {active_df['is_declining_label'].mean():.4f}")

--- QUERY 3: AVAILABILITY & SURVIVING ROWS ---
Raw Dataset Rows: 30,000
Active Slice (impressions_90d > 0 AND content_age_days >= 90): 30,000
Retention Rate: 100.00%
Active Slice Declining Label Ratio: 0.5421


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Part A: 5-Feature Decision Frame

Below we construct a clean 5-feature frame and provide a one-line justification for why each is knowable at the decision moment:

1. **`impressions_90d`**: Knowable at decision moment because search visibility is recorded in historical trailing logs.
2. **`sessions_90d`**: Knowable at decision moment because user session count is logged prior to review time.
3. **`avg_position`**: Knowable at decision moment because Google Search Console rank position is observed historically.
4. **`ctr`**: Knowable at decision moment because historical clicks divided by impressions is measured before the decision.
5. **`content_age_days`**: Knowable at decision moment because publication timestamp is fixed in content metadata.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

# Prepare dataset slice
features_honest = ['impressions_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days']
X_honest = active_df[features_honest].fillna(0)
y = active_df['is_declining_label']

# Simple Train/Test Split (Client Holdout)
clients = active_df['client_id'].unique()
np.random.seed(42)
train_clients = np.random.choice(clients, size=int(len(clients) * 0.75), replace=False)

train_mask = active_df['client_id'].isin(train_clients)
test_mask = ~train_mask

X_train_h, y_train = X_honest[train_mask], y[train_mask]
X_test_h, y_test = X_honest[test_mask], y[test_mask]

# Train Honest Model
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
rf_honest.fit(X_train_h, y_train)

preds_honest_prob = rf_honest.predict_proba(X_test_h)[:, 1]
auc_honest = roc_auc_score(y_test, preds_honest_prob)

# Precision@50 calculation
top50_idx_h = np.argsort(preds_honest_prob)[-50:]
precision50_honest = y_test.iloc[top50_idx_h].mean()

print("--- 5-FEATURE HONEST MODEL RESULTS ---")
print(f"Honest Model ROC-AUC: {auc_honest:.4f}")
print(f"Honest Model Precision@50: {precision50_honest:.4f}")

--- 5-FEATURE HONEST MODEL RESULTS ---
Honest Model ROC-AUC: 0.6712
Honest Model Precision@50: 0.6000


### Part B: The Leakage Trap (Deliberate Feature Leakage Experiment)

Now we spring the trap on purpose: we inject `trend_pct` (a feature derived from the same measurement window as `trend_direction` / `is_declining_label`).

In [14]:
# Injected Leaked Feature Frame
X_leaked = active_df[features_honest + ['trend_pct']].fillna(0)

X_train_l = X_leaked[train_mask]
X_test_l = X_leaked[test_mask]

rf_leaked = RandomForestClassifier(n_estimators=100, random_state=42)
rf_leaked.fit(X_train_l, y_train)

preds_leaked_prob = rf_leaked.predict_proba(X_test_l)[:, 1]
auc_leaked = roc_auc_score(y_test, preds_leaked_prob)

top50_idx_l = np.argsort(preds_leaked_prob)[-50:]
precision50_leaked = y_test.iloc[top50_idx_l].mean()

print("--- LEAKED MODEL RESULTS (THE TRAP) ---")
print(f"Leaked Model ROC-AUC: {auc_leaked:.4f} (Artificial perfection!)")
print(f"Leaked Model Precision@50: {precision50_leaked:.4f}")

# Comparison Table
leakage_comparison = {
    "Model Variant": ["Leaked Model (with trend_pct)", "Honest Model (5 safe features)"],
    "ROC-AUC": [f"{auc_leaked:.4f}", f"{auc_honest:.4f}"],
    "Precision@50": [f"{precision50_leaked:.4f}", f"{precision50_honest:.4f}"],
    "Verdict": ["FATAL LEAKAGE - Model cheats by reading target definition", "SAFE & HONEST - Real decision utility"]
}

pd.DataFrame(leakage_comparison)

--- LEAKED MODEL RESULTS (THE TRAP) ---
Leaked Model ROC-AUC: 1.0000 (Artificial perfection!)
Leaked Model Precision@50: 1.0000


,Model Variant,ROC-AUC,Precision@50,Verdict
0,Leaked Model (with trend_pct),1.0000,1.0000,FATAL LEAKAGE - Model cheats by reading target...
1,Honest Model (5 safe features),0.6712,0.6000,SAFE & HONEST - Real decision utility


### Part C: Data Limitations

*What can this data never tell you?*

1. **Unbalanced Client History:** Different clients entered tracking at different dates. GA4 engagement metrics may be uncollected for older historical periods (`ga4_data_available = FALSE`).
2. **Snapshot vs. Forward Window:** Trailing 90-day snapshots provide a proxy label (`trend_direction`). Proving true causal recovery requires non-overlapping forward-looking target windows ($[T+1, T+30]$).
3. **No Causal Guarantee:** A page flagged for refresh does not guarantee traffic recovery upon edit; it identifies pages with high decision utility for human review.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.